# Week 1: Runway Artifact Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/elenaajayi/spec-gap-activation-probe/blob/main/notebooks/05_week1_runway_artifact_review.ipynb)

This notebook consolidates the completed pre-fellowship activation-probe artifacts. It does not rerun generation or extract new activations. Use it to inspect the saved responses, summarize the probe metrics, and write the Week 1 measurement claim carefully.

In [0]:
from pathlib import Path
import json
import os
import sys

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    try:
        import google.colab  # type: ignore
        IN_COLAB = True
    except Exception:
        IN_COLAB = False

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
    print("Drive mounted. This notebook reads saved artifacts directly from Drive and does not need to clone the repo.")

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root))

candidates = []
if os.environ.get("SPEC_GAP_ARTIFACT_ROOT"):
    candidates.append(Path(os.environ["SPEC_GAP_ARTIFACT_ROOT"]))
candidates += [
    repo_root / "artifacts",
    Path.home() / "Downloads" / "artifacts",
    Path("/content/drive/MyDrive/spec-gap-activation-probe/artifacts"),
]

artifact_root = next((p for p in candidates if (p / "02_collusion_probe").exists()), None)
if artifact_root is None:
    searched = "\n".join(str(p) for p in candidates)
    raise FileNotFoundError(
        "Could not find 02_collusion_probe/. In Colab, put the artifact folder at "
        "/content/drive/MyDrive/spec-gap-activation-probe/artifacts/02_collusion_probe "
        "or set SPEC_GAP_ARTIFACT_ROOT before running this cell.\n\nSearched:\n" + searched
    )

collusion_dir = artifact_root / "02_collusion_probe"
analysis_dir = artifact_root / "03_analysis"
print("repo_root =", repo_root)
print("artifact_root =", artifact_root)

In [0]:
required = [
    "week2_collusion_probe_results.json",
    "week2_collusion_probe_responses.json",
    "week2_collusion_probe_activations.npz",
    "week2_ep_results.json",
]

artifact_table = [
    {
        "file": name,
        "exists": (collusion_dir / name).exists(),
        "size_mb": round((collusion_dir / name).stat().st_size / 1_000_000, 3) if (collusion_dir / name).exists() else None,
        "path": str(collusion_dir / name),
    }
    for name in required
]
artifact_table
assert all(row["exists"] for row in artifact_table), "Missing required runway artifacts."

In [0]:
results = json.loads((collusion_dir / "week2_collusion_probe_results.json").read_text())
responses = json.loads((collusion_dir / "week2_collusion_probe_responses.json").read_text())
ep = json.loads((collusion_dir / "week2_ep_results.json").read_text())

setup = {
    key: results[key]
    for key in ["experiment", "model", "date", "n_scenarios", "n_prompts", "n_colluder", "n_honest", "token_position", "max_new_tokens", "decoding", "layers", "source_dataset", "pca_components"]
}
setup

In [0]:
metric_rows = []
for layer in map(str, results["layers"]):
    strat = results["stratified_cv"][layer]
    lso = results["leave_scenario_out_cv"][layer]
    metric_rows.append({
        "layer": int(layer),
        "stratified_auroc_mean": strat["auroc_mean"],
        "stratified_auroc_std": strat["auroc_std"],
        "accuracy_mean": strat["accuracy_mean"],
        "brier_mean": strat["brier_mean"],
        "ece_mean": strat["ece_mean"],
        "lso_auroc_mean": lso["auroc_mean"],
        "lso_auroc_std": lso["auroc_std"],
    })

metrics = sorted(metric_rows, key=lambda row: row["stratified_auroc_mean"], reverse=True)
metrics
print("Best layer by fold-mean stratified AUROC:", metrics[0]["layer"])
print("Communication note: the Week 3 report/presentation use pooled AUROC with bootstrap intervals, so keep metric conventions explicit.")

In [0]:
response_rows = []
for row in responses:
    meta = row["metadata"]
    response_rows.append({
        "prompt_index": row["prompt_index"],
        "label": row["label"],
        "role": meta.get("role"),
        "scenario_id": meta.get("scenario_id"),
        "domain": meta.get("domain"),
        "agent_name": meta.get("agent_name"),
        "response_words": len(row["response"].split()),
        "response": row["response"],
    })

from collections import Counter
role_label_counts = Counter((row["role"], row["label"]) for row in response_rows)
domain_counts = Counter(row["domain"] for row in response_rows)
print("role/label counts:", role_label_counts)
print("domain counts:", domain_counts)
response_rows[:8]

out_csv = collusion_dir / "week1_response_review.csv"
import csv
with out_csv.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(response_rows[0].keys()))
    writer.writeheader()
    writer.writerows(response_rows)
print("wrote", out_csv)

In [0]:
ep_rows = []
for layer, info in ep["layers"].items():
    sig = [p for p in info["partitions"] if p.get("fisher_p", 1.0) < 0.05]
    best = sorted(sig, key=lambda x: x["fisher_p"])[0] if sig else None
    ep_rows.append({
        "layer": int(layer),
        "n_partitions": info["n_partitions"],
        "n_sig_partitions": len(sig),
        "best_size": best["size"] if best else None,
        "best_collusion_rate": best["collusion_rate"] if best else None,
        "best_fisher_p": best["fisher_p"] if best else None,
        "best_cos_probe": best["cos_probe"] if best else None,
        "best_cos_dim": best["cos_dim"] if best else None,
    })
ep_rows

## Week 1 interpretation

The runway work shows that the measurement stack works: prompts were generated, responses were saved, residual-stream activations were extracted, probes were trained, calibration was checked, and geometry was inspected. It also shows that the simple linear probe is not enough for the final SPEC-GAP detector. The signal is weak, no single layer is defensible as the layer, leave-scenario-out transfer is unstable, and calibration is not strong enough for probability-threshold monitoring.

The fellowship should carry forward a small set of candidate layers and move to trajectory-aware evaluation over planner-worker-executor exploit chains.